# SpaceGen GPU worker: splat -> per-object meshes

Runs `backend/gpu_worker` on a CUDA runtime (Colab T4/A100 or any NVIDIA box). See `docs/decisions/002-splat-to-editable-objects.md`.

1. Runtime -> Change runtime type -> GPU.
2. Run cells 1-3 (install + CUDA smoke test on the synthetic room).
3. Cell 4 runs the real pipeline on an uploaded splat; cell 5 serves the job API through a Cloudflare quick tunnel for the local backend.

**Status:** written on a Windows/AMD machine without CUDA; these cells have not been executed yet.

In [ ]:
#@title 1 - Check GPU, fetch code
!nvidia-smi -L
REPO_URL = 'https://github.com/tanishq-sol/space-gen.git'  #@param {type:"string"}
BRANCH = 'spike/gpu-object-meshing'  #@param {type:"string"}
import os
if not os.path.exists('/content/space-gen'):
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/space-gen
%cd /content/space-gen/backend

In [ ]:
#@title 2 - Install worker deps (~3-5 min; gsplat compiles its CUDA kernels on first use)
!pip install -q -r requirements-gpu-worker.txt
import torch, gsplat
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))
print('gsplat', gsplat.__version__)

In [ ]:
#@title 3 - CUDA smoke test: synthetic room, gsplat renderer, fake masks (compare with the CPU test numbers)
import sys, time, numpy as np
sys.path.insert(0, '.')
from gpu_worker.cameras import room_cameras
from gpu_worker.render import GsplatRenderer
from gpu_worker.segment import segment, shell_prior
from gpu_worker.synthetic import CEILING, INTERPRETATION, OUTLINE, FakeMaskDetector, build_scene, precision_recall
splat, gt = build_scene()
r = GsplatRenderer()
cams = room_cameras(OUTLINE, CEILING, width=192, height=144)
t = time.time()
dets = [FakeMaskDetector(splat, gt, r).detect_view(c) for c in cams]
seg = segment(splat, cams, dets, r, bg_prior=shell_prior(splat, INTERPRETATION))
print(f'{time.time() - t:.1f}s')
for i in seg.instances:
    k = np.bincount(gt[i.indices]).argmax()
    print(i.label, [round(x, 3) for x in precision_recall(i.indices, gt, k)])
# CPU reference (pytest): contribution P=0.995/0.995/0.995, R=1.000/1.000/0.922

In [ ]:
#@title 4 - Real run: upload a .ply splat, T10 room structure, segmentation + meshing
import dataclasses, json, time
from pathlib import Path
from gpu_worker.pipeline import run_pipeline
from gpu_worker.splat import load_ply, transform
from services.object_detection.detectors import GroundedSamDetector
from services.room_structure import RoomStructureOptions, analyze_room
SPLAT = '/content/scene.ply'  #@param {type:"string"}
UP_HINT = (0.0, -1.0, 0.0)  # 3DGS/COLMAP exports are usually -Y up; test scene.ply needed this
if not Path(SPLAT).exists():
    from google.colab import files
    up = files.upload(); SPLAT = '/content/' + next(iter(up))
t0 = time.time()
splat = load_ply(SPLAT)
sample = splat.subset(np.random.default_rng(0).choice(len(splat), min(len(splat), 400_000), replace=False))
room = analyze_room(sample.means, sample.opacities, options=RoomStructureOptions(up_hint=UP_HINT))
splat = transform(splat, np.asarray(room.scan_to_scene, float).reshape(4, 4))
interp = json.loads(room.interpretation.model_dump_json())
print(f'{len(splat):,} gaussians, ceiling {interp["ceilingHeight"]:.2f} m, {len(interp["walls"])} walls, {time.time() - t0:.0f}s')
result = run_pipeline(splat, interp, GroundedSamDetector(), GsplatRenderer(), Path('/content/out'),
                      progress=lambda p, s: print(f'{p:3d}% {s} ({time.time() - t0:.0f}s)'))
for o in result['objects']:
    print(o['label'], o['gaussianCount'], o['triangles'], {k: round(v, 2) for k, v in o['dimensions'].items()})

In [ ]:
#@title 5 - Serve the job API for the local backend (Cloudflare quick tunnel)
import os, secrets, subprocess, threading, re, time
import uvicorn
from gpu_worker.server import create_app
TOKEN = os.environ.get('GPU_WORKER_TOKEN') or secrets.token_urlsafe(24)
app = create_app(TOKEN, Path('/content/jobs'))
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8800, log_level='warning'), daemon=True).start()
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null
proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8800'], stderr=subprocess.PIPE, text=True)
for line in proc.stderr:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('GPU_WORKER_URL =', m.group(0)); print('GPU_WORKER_TOKEN =', TOKEN); break